# Chapter 17 — Semantic Search: Finding What You Mean, Not What You Type

**Volume 2: Practical Applications — AI for Networking and Security Engineers**

There is a fundamental problem at the heart of every search system:
**the user rarely knows the exact words that appear in the document they need.**

A network engineer asking *"why is my BGP session not coming up?"* might be searching
for documentation that says *"BGP peer establishment failure due to OPEN message mismatch"*
— same concept, completely different vocabulary. This is the **vocabulary mismatch problem**.

Semantic search solves this by searching for *meaning*, not *words*.

---
**What you will build — 5 hands-on demos:**

| # | Demo | Key concept |
|---|------|-------------|
| 1 | BM25 from Scratch | Why keyword search fails + the BM25 formula |
| 2 | Semantic vs Sparse Search | Vocabulary mismatch — proven with numbers |
| 3 | Query Transformation | HyDE, Step-Back, Query Decomposition with Claude |
| 4 | RAG Fusion | 3 parallel queries + Reciprocal Rank Fusion |
| 5 | Query Router | LLM-based intent routing to the right knowledge base |

> **Prerequisite:** `!pip install anthropic scikit-learn numpy` (run install cell first)


In [ ]:
!pip install -q anthropic scikit-learn numpy

In [ ]:
import os
import anthropic

# ── API Key Setup ─────────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
    print("API key set via input.")

---
## Demo 1 — BM25 from Scratch: The Math Behind Keyword Search

Before semantic search, there was **BM25** (Best Match 25) — the gold standard
for keyword retrieval. Understanding why it fails is the motivation for everything else.

**The BM25 formula:**
```
BM25(t, d) = IDF(t) × [TF(t,d) × (k1 + 1)] / [TF(t,d) + k1 × (1 - b + b × |d|/avgdl)]
```

- `TF(t,d)` = how many times term `t` appears in document `d`
- `IDF(t)` = how rare term `t` is across all documents (rare = important)
- `k1` = saturation (1.2–2.0): stops term frequency from growing infinitely
- `b` = length normalization (0.75): longer docs don't get unfair advantage
- `avgdl` = average document length

**Two improvements over TF-IDF:**
1. **Saturation**: term frequency has diminishing returns (k1 parameter)
2. **Length normalization**: 5 occurrences in 10 sentences > 5 occurrences in 500 sentences

> *In Simple Words: BM25 is a counting machine. It measures how often you used a word.
> It has no idea what the word means. "BGP session down" vs "peer not establishing"
> — zero overlap = zero score, even though they mean the same thing.*


In [ ]:
# Demo 1: BM25 implemented from scratch — understand the formula
import math
import numpy as np
from collections import Counter

# ── Our network documentation corpus ─────────────────────────────────────────
CORPUS = [
    # BGP-related
    "BGP neighbor not establishing: verify AS numbers and check TCP port 179 connectivity.",
    "BGP peer establishment failure due to OPEN message mismatch in AS configuration.",
    "BGP hold timer expired: local 90s remote 180s. Timer mismatch caused session drop.",
    "BGP route not received: inbound route-map filtering prefixes from neighbor.",
    # OSPF-related
    "OSPF adjacency stuck in EXSTART state due to MTU mismatch between peers.",
    "OSPF neighbor in INIT state: hello packets not being received from peer side.",
    "OSPF area 0 backbone must remain contiguous for proper LSA flooding.",
    # Physical layer
    "Interface CRC errors indicate SFP failure or duplex mismatch physical problem.",
    "Duplex mismatch causes late collisions: one side full duplex other side half duplex.",
    # General
    "NTP server unreachable: check routing table and firewall UDP port 123 rules.",
]

def tokenize(text: str) -> list:
    """Simple whitespace tokenizer — lowercase, strip punctuation."""
    import re
    return re.findall(r'[a-z0-9]+', text.lower())

# ── BM25 implementation ───────────────────────────────────────────────────────
class BM25:
    """Full BM25 implementation from the formula in the chapter."""

    def __init__(self, corpus: list, k1: float = 1.5, b: float = 0.75):
        self.corpus    = corpus
        self.k1        = k1    # saturation parameter
        self.b         = b     # length normalization
        self.tokenized = [tokenize(doc) for doc in corpus]
        self.N         = len(corpus)
        self.avgdl     = sum(len(doc) for doc in self.tokenized) / self.N
        self.df        = self._compute_df()

    def _compute_df(self) -> dict:
        """Document frequency: how many docs contain each term."""
        df = Counter()
        for doc in self.tokenized:
            for term in set(doc):
                df[term] += 1
        return df

    def idf(self, term: str) -> float:
        """IDF with BM25 smoothing: log((N - df + 0.5) / (df + 0.5))."""
        df = self.df.get(term, 0)
        return math.log((self.N - df + 0.5) / (df + 0.5) + 1)

    def score(self, query: str, doc_idx: int) -> float:
        """BM25 score for one query against one document."""
        q_terms  = tokenize(query)
        doc      = self.tokenized[doc_idx]
        doc_len  = len(doc)
        tf_count = Counter(doc)
        total    = 0.0
        for term in q_terms:
            tf  = tf_count.get(term, 0)
            idf = self.idf(term)
            # BM25 saturation + normalization formula
            numerator   = tf * (self.k1 + 1)
            denominator = tf + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
            total += idf * (numerator / denominator) if denominator > 0 else 0
        return total

    def search(self, query: str, top_k: int = 3) -> list:
        """Score all documents and return top-k results."""
        scored = [(self.score(query, i), i, self.corpus[i]) for i in range(self.N)]
        scored.sort(reverse=True)
        return scored[:top_k]

# ── Build BM25 index ──────────────────────────────────────────────────────────
bm25 = BM25(CORPUS, k1=1.5, b=0.75)

print(f"BM25 index built:")
print(f"  Documents: {bm25.N}")
print(f"  Avg doc length: {bm25.avgdl:.1f} tokens")
print(f"  Vocabulary size: {len(bm25.df)} unique terms")

# ── Show term IDF values ──────────────────────────────────────────────────────
print("\nTerm rarity (IDF) — higher = more unique to specific docs:")
interesting_terms = ["bgp", "ospf", "mismatch", "neighbor", "session", "the"]
for term in interesting_terms:
    idf_val = bm25.idf(term)
    df_val  = bm25.df.get(term, 0)
    bar = "█" * int(idf_val * 5)
    print(f"  '{term}': IDF={idf_val:.3f} {bar:<20} (in {df_val}/{bm25.N} docs)")

# ── The vocabulary mismatch problem ──────────────────────────────────────────
print("\n" + "="*60)
print("  THE VOCABULARY MISMATCH PROBLEM")
print("="*60)

queries = [
    ("BGP peer not establishing",          "exact match — should find easily"),
    ("BGP session is down",                "paraphrase — 'session' vs 'neighbor'"),
    ("why won't my routing protocol talk", "no technical jargon — BM25 struggles"),
    ("TCP port 179 connection refused",    "specific term — BM25 should find"),
]

for query, note in queries:
    results = bm25.search(query, top_k=1)
    score, idx, doc = results[0]
    print(f'\n  Query: "{query}"')
    print(f"  Note:  {note}")
    print(f"  Best:  [{score:.3f}] {doc[:70]}...")
    if score < 0.5:
        print(f"  ⚠️   Low score — vocabulary mismatch likely!")
    else:
        print(f"  ✅   Good match")


---
## Demo 2 — Semantic vs Sparse: Measuring the Vocabulary Mismatch Gap

**Dense retrieval** (bi-encoder) converts text to vectors capturing *meaning*.
Texts with similar meaning → vectors that are close together in high-dimensional space.

```
"BGP session not coming up"  →  [0.12, -0.45, 0.78, ...]   ← vector A
"BGP peer establishment fail" →  [0.11, -0.43, 0.76, ...]   ← vector B
                                        similarity ≈ 0.97
```

Same concept, different words → similar vectors. BM25 gives 0.

**The two-stage pipeline (production approach):**
```
Stage 1: Sparse BM25     → top-100 candidates  (fast, maximizes recall)
Stage 2: Dense re-ranking → top-5 results       (slow, maximizes precision)
```

> *Networking analogy: bi-encoder = GPS coordinates for meaning.
> "BGP session down" and "peer not establishing" land in the same neighborhood
> on the semantic map, even though they share zero words.*


In [ ]:
# Demo 2: Compare BM25 vs TF-IDF vs semantic (simulated) on the same queries
import math
import numpy as np
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Same corpus from Demo 1
CORPUS = [
    "BGP neighbor not establishing: verify AS numbers and check TCP port 179 connectivity.",
    "BGP peer establishment failure due to OPEN message mismatch in AS configuration.",
    "BGP hold timer expired: local 90s remote 180s. Timer mismatch caused session drop.",
    "BGP route not received: inbound route-map filtering prefixes from neighbor.",
    "OSPF adjacency stuck in EXSTART state due to MTU mismatch between peers.",
    "OSPF neighbor in INIT state: hello packets not being received from peer side.",
    "OSPF area 0 backbone must remain contiguous for proper LSA flooding.",
    "Interface CRC errors indicate SFP failure or duplex mismatch physical problem.",
    "Duplex mismatch causes late collisions: one side full duplex other side half duplex.",
    "NTP server unreachable: check routing table and firewall UDP port 123 rules.",
]

# ── TF-IDF (our "semantic" baseline — simple version of dense retrieval) ───────
vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
tfidf_matrix = vectorizer.fit_transform(CORPUS)

def tfidf_search(query: str, top_k: int = 3) -> list:
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:top_k]
    return [(CORPUS[i], s) for i, s in ranked]

# ── BM25 (reusing from Demo 1) ────────────────────────────────────────────────
import re

def tokenize(text):
    return re.findall(r'[a-z0-9]+', text.lower())

class BM25Simple:
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.corpus = corpus
        self.k1 = k1; self.b = b
        self.docs = [tokenize(d) for d in corpus]
        self.N = len(corpus)
        self.avgdl = sum(len(d) for d in self.docs) / self.N
        self.df = Counter(t for doc in self.docs for t in set(doc))

    def score(self, query, idx):
        q = tokenize(query)
        doc = self.docs[idx]
        tf = Counter(doc)
        dl = len(doc)
        return sum(
            math.log((self.N - self.df.get(t,0) + 0.5)/(self.df.get(t,0) + 0.5) + 1)
            * (tf.get(t,0) * (self.k1+1))
            / (tf.get(t,0) + self.k1*(1-self.b+self.b*dl/self.avgdl))
            for t in q if self.df.get(t,0)
        )

    def search(self, query, top_k=3):
        scored = [(self.score(query, i), CORPUS[i]) for i in range(self.N)]
        return sorted(scored, reverse=True)[:top_k]

bm25 = BM25Simple(CORPUS)

# ── Side-by-side comparison ───────────────────────────────────────────────────
test_cases = [
    # (query, description, expected_doc_hint)
    ("BGP session is down",               "Paraphrase — 'session' not in corpus",    "BGP neighbor not establishing"),
    ("peer establishment fails",           "Synonym — both methods tested",           "BGP peer establishment failure"),
    ("routing protocol neighbors refuse",  "No jargon — vague description",           "BGP neighbor not establishing"),
    ("TCP 179 port refused connection",    "Specific term — keyword search wins",     "BGP neighbor not establishing"),
    ("OSPF stuck not progressing",         "Partial paraphrase of EXSTART",           "OSPF adjacency stuck"),
]

print("="*70)
print("  BM25 vs TF-IDF SEMANTIC SEARCH — COMPARISON")
print("="*70)

bm25_wins = 0
tfidf_wins = 0

for query, note, expected_hint in test_cases:
    bm25_results  = bm25.search(query, top_k=1)
    tfidf_results = tfidf_search(query, top_k=1)

    bm25_doc   = bm25_results[0][1] if bm25_results else ""
    bm25_score = bm25_results[0][0] if bm25_results else 0
    tfidf_doc  = tfidf_results[0][0]
    tfidf_score = tfidf_results[0][1]

    # Check which found the expected document
    bm25_correct  = expected_hint.lower() in bm25_doc.lower()
    tfidf_correct = expected_hint.lower() in tfidf_doc.lower()
    if tfidf_correct and not bm25_correct:
        tfidf_wins += 1
    elif bm25_correct and not tfidf_correct:
        bm25_wins += 1

    print(f'\n  Query: "{query}"')
    print(f"  Note:  {note}")
    print(f"  BM25  [{bm25_score:5.3f}] {'✅' if bm25_correct else '❌'} {bm25_doc[:60]}...")
    print(f"  TFIDF [{tfidf_score:5.3f}] {'✅' if tfidf_correct else '❌'} {tfidf_doc[:60]}...")

print(f"\n{'='*70}")
print(f"  RESULT: TF-IDF semantic won {tfidf_wins}× | BM25 keyword won {bm25_wins}×")
print(f"  Conclusion: semantic search handles vocabulary mismatch better.")
print(f"  Best practice: combine BOTH (hybrid search) for maximum recall.")
print(f"{'='*70}")

# ── Show the two-stage architecture ──────────────────────────────────────────
print("\n  TWO-STAGE PRODUCTION PIPELINE:")
print("  Stage 1: BM25 retrieves top-N fast  → maximizes RECALL (don't miss anything)")
print("  Stage 2: Semantic re-ranks top-N    → maximizes PRECISION (best answer at top)")
print("\n  Why not just use semantic for everything?")
print("  BM25 is 1000x faster — feasible on millions of docs.")
print("  Semantic re-ranking on top-50 candidates = fast AND precise.")


---
## Demo 3 — Query Transformation: Ask Better Questions Automatically

Even the best retrieval system fails if the query is vague, jargon-free, or too narrow.
**Query transformation** uses Claude to rewrite the query before searching.

### Three techniques:

**1. HyDE (Hypothetical Document Embeddings)**
The query and the document live in different linguistic registers.
A question is short and informal. A runbook is formal and dense.
HyDE generates a *hypothetical answer* and searches with that instead.

**2. Step-Back Prompting**
The specific question is too narrow. The documentation covers the principle.
Step-back asks: "what is the underlying concept behind this specific question?"
Then searches for that concept.

**3. Query Decomposition**
Complex questions contain multiple information needs.
Decompose into sub-queries, retrieve for each, synthesize the final answer.

> *Networking analogy: HyDE is like policy-based routing —
> don't route by the source question, route by the destination answer-shape.
> Step-back is like going up the route table hierarchy to find the summary route.*


In [ ]:
# Demo 3: Three query transformation techniques with Claude
import anthropic
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

client = anthropic.Anthropic()
HAIKU  = "claude-haiku-4-5-20251001"
SONNET = "claude-sonnet-4-20250514"

# ── Knowledge base ────────────────────────────────────────────────────────────
KB = [
    "BGP OPEN message mismatch: AS number, hold timer, or BGP version mismatch causes session rejection.",
    "BGP neighbor authentication: MD5 password must match on both peers. Mismatch causes TCP RST.",
    "BGP hold timer: if hold timer expires because keepalives stop, the session drops with Notification code 4.",
    "BGP Notification codes: 1=Header Error, 2=OPEN Message Error, 3=UPDATE Message Error, 4=Hold Timer Expired.",
    "OSPF hello/dead interval mismatch: both sides must match on the same segment. Check: show ip ospf interface.",
    "OSPF MTU mismatch: causes EXSTART adjacency failure. Fix: ip ospf mtu-ignore or match MTU on both sides.",
    "OSPF authentication: must match type (null/simple/MD5) and key. Area authentication overrides interface.",
    "Subnet mask mismatch on a point-to-point link causes OSPF to reject the neighbor.",
    "BGP confederations: divide large AS into sub-ASes. eBGP rules apply internally between sub-ASes.",
    "BGP route reflectors: eliminate full-mesh iBGP requirement. Clients don't need sessions with each other.",
]
texts = KB[:]

vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
matrix = vectorizer.fit_transform(texts)

def vector_search(query: str, top_k: int = 3) -> list:
    q = vectorizer.transform([query])
    scores = cosine_similarity(q, matrix).flatten()
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:top_k]
    return [(KB[i], s) for i, s in ranked]

def llm_call(prompt: str, max_tokens: int = 200) -> str:
    resp = client.messages.create(
        model=HAIKU, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.content[0].text.strip()

# ─────────────────────────────────────────────────────────────────────────────
# Technique 1: HyDE — Hypothetical Document Embeddings
# ─────────────────────────────────────────────────────────────────────────────
def hyde_search(question: str, top_k: int = 2) -> tuple:
    """Generate hypothetical answer, search with that instead of the question."""
    prompt = (
        f"You are a senior network engineer. Write a 2-3 sentence technical paragraph "
        f"that could appear in a networking runbook as the answer to this question. "
        f"Use technical terminology. Do not say 'this document explains':\n\n{question}"
    )
    hypothetical = llm_call(prompt)
    results = vector_search(hypothetical, top_k=top_k)
    return hypothetical, results

# ─────────────────────────────────────────────────────────────────────────────
# Technique 2: Step-Back Prompting
# ─────────────────────────────────────────────────────────────────────────────
def stepback_search(question: str, top_k: int = 2) -> tuple:
    """Identify the underlying principle, search for that."""
    prompt = (
        f"This is a very specific networking question. Identify the underlying "
        f"networking concept or principle that would help answer it. "
        f"Respond with just one sentence describing the broader concept:\n\n{question}"
    )
    principle = llm_call(prompt, max_tokens=80)
    results = vector_search(principle, top_k=top_k)
    return principle, results

# ─────────────────────────────────────────────────────────────────────────────
# Technique 3: Query Decomposition
# ─────────────────────────────────────────────────────────────────────────────
def decompose_search(question: str, top_k: int = 2) -> tuple:
    """Break complex question into sub-queries, search each."""
    prompt = (
        f"Break this complex networking question into 2-3 simpler sub-questions. "
        f"Respond with ONLY a numbered list, one sub-question per line:\n\n{question}"
    )
    sub_questions_text = llm_call(prompt, max_tokens=150)
    sub_questions = [
        line.lstrip("0123456789.-) ").strip()
        for line in sub_questions_text.splitlines()
        if len(line.strip()) > 10
    ][:3]

    all_results = {}
    for sq in sub_questions:
        for doc, score in vector_search(sq, top_k=2):
            all_results[doc] = max(all_results.get(doc, 0), score)

    top_docs = sorted(all_results.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return sub_questions, top_docs

# ── Run all three on test queries ─────────────────────────────────────────────
print("="*65)
print("  QUERY TRANSFORMATION DEMO")
print("="*65)

test_queries = [
    ("My routing protocol won't talk to its neighbor",
     "Vague, no jargon — all techniques should help"),
    ("OSPF neighbor stuck not going past EXSTART, what are all the possible causes?",
     "Specific symptom — step-back and decompose should help"),
]

for question, note in test_queries:
    print(f'\n  Original Question: "{question}"')
    print(f"  Note: {note}")
    print(f"\n  ── Direct Search (no transformation) ──")
    for doc, score in vector_search(question, top_k=1):
        print(f"    [{score:.3f}] {doc[:75]}...")

    print(f"\n  ── HyDE Search ──")
    hypo, res = hyde_search(question, top_k=1)
    print(f'    Hypothetical: "{hypo[:100]}..."')
    for doc, score in res:
        print(f"    [{score:.3f}] {doc[:75]}...")

    print(f"\n  ── Step-Back Search ──")
    principle, res = stepback_search(question, top_k=1)
    print(f'    Principle: "{principle[:100]}"')
    for doc, score in res:
        print(f"    [{score:.3f}] {doc[:75]}...")

    print(f"\n  ── Decomposed Search ──")
    sub_qs, res = decompose_search(question, top_k=2)
    print(f"    Sub-questions: {sub_qs}")
    for doc, score in res:
        print(f"    [{score:.3f}] {doc[:75]}...")


---
## Demo 4 — RAG Fusion: Multiple Parallel Queries + Reciprocal Rank Fusion

**RAG Fusion** = generate multiple query variations → retrieve for each → merge with RRF.

Why does this work?
- A document at rank 3 in **three** different query result lists is almost certainly
  more relevant than a document at rank 1 in only **one** list.
- RRF "votes" — documents that multiple queries independently find get promoted.

**The RRF formula:**
```
Score(D) = Σ  1 / (k + rank_i(D))
```
Where `k=60` (constant preventing extreme ranks from dominating)
and the sum is over all query lists.

**Why k=60?** It makes rank differences meaningful but not catastrophic.
The difference between rank 1 and rank 2 = `1/61 - 1/62 ≈ 0.0003`
The difference between rank 1 and rank 61 = `1/61 - 1/121 ≈ 0.008`

> *In Simple Words: RAG Fusion is like asking the same question to three colleagues
> in three different ways. Any document that multiple people independently found
> relevant gets promoted. More robust than trusting any single query.*


In [ ]:
# Demo 4: RAG Fusion — parallel queries + Reciprocal Rank Fusion
import anthropic
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

client = anthropic.Anthropic()
HAIKU = "claude-haiku-4-5-20251001"

# ── Network knowledge base ────────────────────────────────────────────────────
KB = [
    {"id": "K01", "text": "BGP neighbor state IDLE: session never established. Check AS numbers and TCP port 179."},
    {"id": "K02", "text": "BGP OPEN message rejection: AS number mismatch, hold timer mismatch, or BGP version mismatch."},
    {"id": "K03", "text": "BGP MD5 authentication: password must match on both peers. Mismatch causes silent TCP RST."},
    {"id": "K04", "text": "BGP hold timer mismatch: local 90s remote 180s. Session drops with Hold Timer Expired code 4."},
    {"id": "K05", "text": "BGP Notification message code 2 subcode 2: Bad Peer AS. The remote AS does not match configuration."},
    {"id": "K06", "text": "OSPF EXSTART state: Database Description exchange fails. Usually MTU mismatch between peers."},
    {"id": "K07", "text": "OSPF hello/dead timer mismatch: both sides must use identical values on same link segment."},
    {"id": "K08", "text": "OSPF area authentication: type and key must match. Area config overrides interface config."},
    {"id": "K09", "text": "Interface CRC errors: physical layer problem. Bad cable, dirty SFP, or duplex mismatch."},
    {"id": "K10", "text": "Duplex mismatch: full-duplex vs half-duplex causes late collisions and link instability."},
]
texts = [k["text"] for k in KB]

# Build index
vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
mat = vec.fit_transform(texts)

def search(query: str, top_k: int = 5) -> list:
    """Returns list of (doc_id, score) tuples."""
    q = vec.transform([query])
    scores = cosine_similarity(q, mat).flatten()
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:top_k]
    return [(KB[i]["id"], scores[i]) for i in [r[0] for r in ranked]]

# ── Generate query variations ─────────────────────────────────────────────────
def generate_query_variations(original: str, n: int = 3) -> list:
    """Use Haiku to rephrase the query in N different ways."""
    prompt = (
        f"Generate {n} different ways to ask the same networking question. "
        f"Each should use different vocabulary and phrasing. "
        f"Respond with ONLY the questions, one per line, no numbering:\n\n"
        f"Original: {original}"
    )
    resp = client.messages.create(
        model=HAIKU, max_tokens=200,
        messages=[{"role": "user", "content": prompt}],
    )
    lines = [
        l.strip().lstrip("0123456789.-) ")
        for l in resp.content[0].text.strip().splitlines()
        if len(l.strip()) > 10
    ]
    return [original] + lines[:n]   # include original + n variations

# ── Reciprocal Rank Fusion ────────────────────────────────────────────────────
def rrf(ranked_lists: list, k: int = 60) -> list:
    """
    Merge multiple ranked lists.
    Each list is [(doc_id, score), ...] sorted by score.
    Returns [(doc_id, rrf_score), ...] sorted by rrf_score.
    """
    scores = {}
    for ranked_list in ranked_lists:
        for rank, (doc_id, _) in enumerate(ranked_list, start=1):
            scores[doc_id] = scores.get(doc_id, 0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

# ── Show RRF formula visually ─────────────────────────────────────────────────
def show_rrf_calculation(doc_id: str, ranked_lists: list, k: int = 60):
    """Show how RRF score is computed for one document."""
    parts = []
    for i, lst in enumerate(ranked_lists):
        ids = [d for d, _ in lst]
        rank = ids.index(doc_id) + 1 if doc_id in ids else None
        if rank:
            contribution = 1.0 / (k + rank)
            parts.append(f"1/({k}+{rank})={contribution:.4f}")
    total = sum(1/(k + ([d for d, _ in lst].index(doc_id)+1)) for lst in ranked_lists if doc_id in [d for d, _ in lst])
    return " + ".join(parts) + f" = {total:.4f}"

# ── Full RAG Fusion pipeline ──────────────────────────────────────────────────
def rag_fusion(original_query: str):
    print(f"\n{'='*65}")
    print(f"  RAG FUSION DEMO")
    print(f'  Original query: "{original_query}"')
    print(f"{'='*65}")

    # Step 1: Generate query variations
    queries = generate_query_variations(original_query, n=2)
    print(f"\n  Step 1 — Query variations generated ({len(queries)} total):")
    for i, q in enumerate(queries):
        label = "Original" if i == 0 else f"Variation {i}"
        print(f"    [{label}] {q}")

    # Step 2: Search with each query
    print(f"\n  Step 2 — Search results per query:")
    all_lists = []
    for q in queries:
        results = search(q, top_k=5)
        all_lists.append(results)
        top3 = [f"{doc_id}({score:.2f})" for doc_id, score in results[:3]]
        print(f'    Query "{q[:40]}...\": {top3}')

    # Step 3: RRF merge
    fused = rrf(all_lists)
    print(f"\n  Step 3 — After Reciprocal Rank Fusion:")
    for doc_id, rrf_score in fused[:5]:
        doc_text = next(k["text"] for k in KB if k["id"] == doc_id)
        calc = show_rrf_calculation(doc_id, all_lists)
        print(f"    [{doc_id}] RRF={rrf_score:.4f}  ({calc})")
        print(f"           {doc_text[:70]}...")

    # Step 4: Compare to single-query result
    single = search(original_query, top_k=3)
    print(f"\n  Comparison:")
    print(f"    Single query top result: {single[0][0]} ({single[0][1]:.3f})")
    print(f"    RAG Fusion top result:   {fused[0][0]} ({fused[0][1]:.4f} RRF)")


rag_fusion("BGP session keeps dropping, what are all the possible causes?")


---
## Demo 5 — Query Router: Sending Questions to the Right Knowledge Base

In real deployments you have **multiple knowledge bases**:
- Vendor documentation (reference material)
- Internal runbooks (troubleshooting procedures)
- Configuration archives (device configs)
- Incident tickets (past incidents and resolutions)

Not every query should search all of them — that wastes compute and adds noise.

**Query routing** uses intent classification to direct queries to the right source.

```
"What is the max prefix limit for BGP?"    → vendor_docs  (reference question)
"My BGP session keeps dropping"            → runbooks     (troubleshooting question)
"What's configured on core-01 for BGP?"   → config_db    (config lookup)
"Has this OSPF issue happened before?"    → incidents    (historical search)
```

> *Networking analogy: Query routing is like BGP route selection.
> Different question types go to different next-hops (knowledge sources).
> The router (Claude) reads the query and selects the best path.*


In [ ]:
# Demo 5: Query router — LLM-based intent classification to right knowledge source
import anthropic
import numpy as np
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

client = anthropic.Anthropic()
HAIKU  = "claude-haiku-4-5-20251001"
SONNET = "claude-sonnet-4-20250514"

# ── Four separate knowledge bases ─────────────────────────────────────────────
KNOWLEDGE_BASES = {
    "vendor_docs": [
        "BGP maximum-prefix: default limit is 1000 prefixes per neighbor on IOS-XE.",
        "BGP hold timer range: 3-65535 seconds. Minimum 3 seconds recommended.",
        "OSPF SPF delay: initial 50ms, minimum 200ms, maximum 5000ms by default.",
        "OSPF supports 255 areas per router process on Cisco IOS platform.",
        "BGP supports 4-byte AS numbers (RFC 6793). Range: 1 to 4,294,967,295.",
    ],
    "runbooks": [
        "BGP troubleshooting: start with show bgp summary. Check state column for non-Established peers.",
        "When BGP state is IDLE: verify TCP 179 reachable, check AS numbers, verify routing to peer.",
        "BGP hold timer mismatch fix: configure matching timers with 'neighbor X timers keepalive hold'.",
        "OSPF EXSTART fix: check MTU with 'show interface'. Apply 'ip ospf mtu-ignore' if MTU mismatches.",
        "Interface flapping procedure: check physical first. Verify SFP, cable, and duplex settings.",
    ],
    "config_db": [
        "core-01 BGP config: router bgp 65001, neighbor 10.1.1.2 remote-as 65002, timers 30 90.",
        "core-01 OSPF config: router ospf 1, router-id 10.0.0.1, area 0 authentication message-digest.",
        "core-02 BGP config: router bgp 65001, neighbor 10.1.1.1 remote-as 65001, route-reflector-client.",
        "branch-01 BGP config: router bgp 65001, neighbor 203.0.113.2 remote-as 65000, timers 10 30.",
        "core-01 interface Gi0/0: ip address 203.0.113.1/30, duplex full, speed 1000.",
    ],
    "incidents": [
        "INC-2024-001: BGP to ISP down. Root cause: SFP failure GigabitEthernet0/0. Fix: SFP replaced.",
        "INC-2024-003: OSPF flapping core-01 to dist-01. Root cause: MTU mismatch. Fix: mtu-ignore applied.",
        "INC-2024-007: BGP hold timer expired repeatedly. Root cause: ISP changed timers. Fix: timers matched.",
        "INC-2024-009: Interface Gi0/1 CRC errors. Root cause: Duplex mismatch with server. Fix: duplex full.",
        "INC-2024-012: BGP prefix limit hit. Upstream started advertising extra prefixes. Fix: limit raised.",
    ],
}

# Build one TF-IDF index per knowledge base
indexes = {}
for name, docs in KNOWLEDGE_BASES.items():
    v = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
    m = v.fit_transform(docs)
    indexes[name] = (v, m, docs)

def search_kb(kb_name: str, query: str, top_k: int = 2) -> list:
    v, m, docs = indexes[kb_name]
    q_vec = v.transform([query])
    scores = cosine_similarity(q_vec, m).flatten()
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:top_k]
    return [(docs[i], scores[i]) for i in [r[0] for r in ranked]]

# ── LLM-based query router ────────────────────────────────────────────────────
ROUTE_DESCRIPTIONS = {
    "vendor_docs": "Technical reference questions: capabilities, limits, defaults, protocol specifications",
    "runbooks":    "Troubleshooting and how-to questions: how to fix, what to check, procedure questions",
    "config_db":   "Configuration lookup: what is configured on a specific device, current settings",
    "incidents":   "Historical questions: has this happened before, past incidents, previous outages",
}

def classify_intent(query: str) -> dict:
    """Use Claude Haiku to classify query intent and select knowledge base."""
    kb_list = "\n".join(f'- "{k}": {v}' for k, v in ROUTE_DESCRIPTIONS.items())
    prompt = (
        f"Classify this network question and route it to the best knowledge base.\n\n"
        f"Question: {query}\n\n"
        f"Available knowledge bases:\n{kb_list}\n\n"
        f"Respond with valid JSON only:\n"
        f'{{"route": "kb_name", "reason": "one sentence why", "confidence": 0.0-1.0}}'
    )
    resp = client.messages.create(
        model=HAIKU, max_tokens=120,
        messages=[{"role": "user", "content": prompt}],
    )
    raw = resp.content[0].text.strip()
    try:
        import re
        match = re.search(r'\{[\s\S]*\}', raw)
        result = json.loads(match.group()) if match else {}
        if result.get("route") not in KNOWLEDGE_BASES:
            result["route"] = "runbooks"   # safe default
        return result
    except Exception:
        return {"route": "runbooks", "reason": "classification failed", "confidence": 0.5}

def routed_search(query: str) -> dict:
    """Classify the query, route to the right KB, search, and generate answer."""
    # Step 1: Classify
    classification = classify_intent(query)
    route      = classification.get("route", "runbooks")
    reason     = classification.get("reason", "")
    confidence = classification.get("confidence", 0)

    # Step 2: Search the right KB
    results = search_kb(route, query, top_k=2)

    # Step 3: Generate answer from retrieved context
    context = "\n".join(f"[{i+1}] {doc}" for i, (doc, _) in enumerate(results))
    system  = (
        "You are a network operations assistant. Answer using ONLY the provided context. "
        "Be concise (2-3 sentences). Cite source [1] or [2]."
    )
    resp = client.messages.create(
        model=SONNET, max_tokens=150, system=system,
        messages=[{"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}],
    )

    return {
        "query":      query,
        "route":      route,
        "reason":     reason,
        "confidence": confidence,
        "sources":    [(doc[:70], score) for doc, score in results],
        "answer":     resp.content[0].text.strip(),
    }

# ── Test the router on diverse questions ──────────────────────────────────────
test_questions = [
    "What is the default BGP hold timer value?",
    "My BGP session to the ISP keeps going down, what should I check?",
    "What BGP configuration is on core-01?",
    "Has there been a BGP issue with the ISP link before?",
]

print("="*65)
print("  QUERY ROUTER DEMO")
print("="*65)

for q in test_questions:
    result = routed_search(q)
    print(f'\n  Question: "{result["query"]}"')
    print(f"  Route:    [{result['route']}]  (confidence: {result['confidence']:.0%})")
    print(f"  Reason:   {result['reason']}")
    print(f"  Sources:  {[s[:45] for s, _ in result['sources']]}")
    print(f"  Answer:   {result['answer'][:150]}")


---
## Summary — What You Built

A complete semantic search system, from keyword search fundamentals to intelligent query routing:

### Demo progression:
1. **BM25 from Scratch** — built the formula, proved vocabulary mismatch (0 score for paraphrases)
2. **Semantic vs Sparse** — measured the gap: TF-IDF semantic handles paraphrases, BM25 handles exact terms
3. **Query Transformation** — three Claude-powered techniques: HyDE (answer-shape search), Step-Back (find the principle), Decomposition (break complex questions)
4. **RAG Fusion** — generate 3 query variations → search each → merge with RRF → more robust than any single query
5. **Query Router** — Claude Haiku classifies intent → routes to vendor_docs / runbooks / config_db / incidents

### Key formulas:

**BM25 saturation:**
```
BM25(t,d) = IDF(t) × [TF × (k1+1)] / [TF + k1 × (1-b + b×|d|/avgdl)]
```
- k1=1.5 (saturation), b=0.75 (length normalization)

**Reciprocal Rank Fusion:**
```
RRF(d) = Σ  1/(60 + rank_i(d))     for each query list i
```
- k=60 prevents extreme ranks from dominating
- Documents appearing in multiple lists get boosted

### When to use each technique:

| Query type | Best approach |
|-----------|--------------|
| Exact terminology (error codes, IDs) | BM25 keyword search |
| Paraphrased / conceptual | Dense semantic search |
| Vague or no jargon | HyDE transformation |
| Very specific symptom | Step-back to principle |
| Multi-part questions | Query decomposition |
| General questions | RAG Fusion (multiple variations) |
| Mixed question types | Query routing + hybrid search |

---
*Chapter 17 — Semantic Search | AI for Networking and Security Engineers*
